# 3.14 Summary statistics table

Summary statistics for the experimental design / sample section of the paper.

Sample: lab groups with both BL and EL data (i.e. excluding attrited lab groups) - the same sample as the balance table (5_1). All variables are measured at baseline.

Continuous variables report mean, SD, min, median, max; 0/1 variables report the share (mean) only. N is the number of lab groups with a non-missing value; missing values are left out of the statistics. Publication counts exist only for the lab groups that consented to data merging, since only their researchers were matched to publications.

## Set-up

In [1]:
# Set-up
import pandas as pd
import numpy as np
import sys
from pathlib import Path
CODE_ROOT = Path.cwd().parents[1]
sys.path.append(str(CODE_ROOT))
import config

In [2]:
# Load data
df = pd.read_csv(
    config.CLEAN_DATA / "final_dataset.csv",
    keep_default_na=False, # Keep "None" as a string, not NaN
    na_values=[""] # Only treat empty strings as NaN
)
panel = pd.read_csv(
    config.PROCESSED_DATA / "panel_processed_5.csv",
    keep_default_na=False,
    na_values=[""],
    usecols=["labgroupid", "survey", "equipment", "number"]
)
publications = pd.read_csv(
    config.PUBLICATON_DATA / "2_Processed" / "publications_matched.csv",
    usecols=["relation", "matched_labgroupids"]
)

## (1) Prepare data

In [3]:
# Keep only lab groups with both BL and EL data (same as the balance table), then take one BL row per lab group
n_surveys = df.groupby("labgroupid")["survey"].nunique()
labgroups_to_keep = n_surveys[n_surveys == 2].index
bl = (
    df[df["labgroupid"].isin(labgroups_to_keep) & (df["survey"] == "BL")]
    .drop_duplicates("labgroupid")
    .set_index("labgroupid")
)
assert len(bl) == len(labgroups_to_keep) # every kept lab group has a BL row
print(f"{len(bl)} of {df['labgroupid'].nunique()} lab groups surveyed at baseline have both BL and EL data")

# d holds every variable shown in the table, one row per lab group (NaN = missing)
d = pd.DataFrame(index=bl.index)

109 of 138 lab groups surveyed at baseline have both BL and EL data


In [4]:
# Sample composition
d["science"] = (bl["faculty"] == "Faculty of Science (MNF)").astype(float)
d["medicine"] = (bl["faculty"] == "Faculty of Medicine (MeF)").astype(float)
d["joint"] = (bl["faculty"] == "Both MNF and MeF").astype(float)
assert (d[["science", "medicine", "joint"]].sum(axis=1) == 1).all() # every lab is in exactly one faculty group

d["treated"] = bl["treated"].astype(float)


In [5]:
# Lab size and baseline electricity
d["no_researchers"] = pd.to_numeric(bl["no_researchers"], errors="raise")
d["electricity_total"] = pd.to_numeric(bl["annual_electricity_total"], errors="raise") / 1000 # kWh -> MWh

In [6]:
# Equipment units per lab at BL, 0 for labs with none of that type
equipment_labels = {
    "it": "IT equipment",
    "fc": "Fume cupboards",
    "freezer": "Freezers",
    "ult": "ULT freezers",
    "fridge": "Fridges",
    "incubator": "CO2 incubators",
    "microbio": "Microbiological safety cabinets",
    "glassware": "Glassware drying cabinets",
    "bath": "Water baths",
    "cryostat": "Cryostats",
    "heater": "Block heaters",
}

units = (
    panel[panel["survey"] == "BL"]
    .groupby(["labgroupid", "equipment"])["number"].sum()
    .unstack(fill_value=0)
    .reindex(index=bl.index, columns=list(equipment_labels), fill_value=0)
)
for eq in equipment_labels:
    d[f"units_{eq}"] = units[eq]
d["units_total"] = units.sum(axis=1)

In [7]:
# Matched publications per lab group. Only consenting labs were matched to publications: a
# consenting lab with no match has 0 publications, but a non-consenting lab is missing (not 0)
pairs = publications.assign(labgroupid=publications["matched_labgroupids"].astype(str).str.split(";")).explode("labgroupid")
pairs["labgroupid"] = pairs["labgroupid"].str.strip().astype(int)
n_pubs = pairs.groupby("labgroupid")["relation"].nunique() # unique publications per lab

consenting = bl["consent_data_merge"] == "Yes I consent to this data collection and merging"
d["n_pubs"] = n_pubs.reindex(bl.index).fillna(0).where(consenting)
d["any_pub"] = (d["n_pubs"] > 0).astype(float).where(consenting)

print(f"Consenting lab groups: {consenting.sum()}")

Consenting lab groups: 81


## (2) Summary statistics

In [8]:
# Table layout: panel -> rows of (label, variable, kind, decimals for mean/SD, decimals for min/median/max)
# kind "cont" shows mean, SD, min, median, max; kind "binary" shows the share (mean) only
equip_order = sorted(equipment_labels, key=lambda eq: d[f"units_{eq}"].mean(), reverse=True) # by mean units, as in 8_2

panels = {
    "Sample composition": [
        ("Science",                          "science",  "binary", 3, None),
        ("Medicine",                         "medicine", "binary", 3, None),
        ("Joint",                            "joint",    "binary", 3, None),
        ("Assigned to treatment",            "treated",  "binary", 3, None),
        ("Number of researchers",            "no_researchers", "cont", 2, 0),
    ],
    "Baseline annual energy use (MWh)": [
        ("Total",                            "electricity_total", "cont", 2, 2),
    ],
    "Baseline equipment (units per group)": (
        [(equipment_labels[eq], f"units_{eq}", "cont", 2, 0) for eq in equip_order]
        + [("Total units", "units_total", "cont", 2, 0)]
    ),
    "Publications (consenting groups only)": [
        ("At least one matched publication", "any_pub", "binary", 3, None),
        ("Matched publications",             "n_pubs",  "cont",   2, 0),
    ],
}

In [9]:
# Compute the statistics for every row (missing values dropped, N = number non-missing)
records = []
for panel_label, rows in panels.items():
    for label, var, kind, d_mean, d_range in rows:
        s = d[var].dropna()
        rec = {"panel": panel_label, "label": label, "mean": s.mean(), "N": len(s)}
        if kind == "cont":
            rec.update({"sd": s.std(), "min": s.min(), "median": s.median(), "max": s.max()})
        records.append(rec)

stats = pd.DataFrame(records)[["panel", "label", "mean", "sd", "min", "median", "max", "N"]]
stats.round(3)

,panel,label,mean,sd,min,median,max,N
0,Sample composition,Science,0.706,NaN,NaN,NaN,NaN,109
1,Sample composition,Medicine,0.248,NaN,NaN,NaN,NaN,109
2,Sample composition,Joint,0.046,NaN,NaN,NaN,NaN,109
3,Sample composition,Assigned to treatment,0.514,NaN,NaN,NaN,NaN,109
4,Sample composition,Number of researchers,12.376,21.697,1.000,8.000,214.000,109
5,Baseline annual energy use (MWh),Total,22.026,36.951,0.232,13.309,300.973,109
6,Baseline equipment (units per group),IT equipment,14.972,23.680,1.000,9.000,230.000,109
7,Baseline equipment (units per group),Fume cupboards,3.596,8.369,0.000,1.000,59.000,109
8,Baseline equipment (units per group),Fridges,3.560,5.508,0.000,2.000,45.000,109
9,Baseline equipment (units per group),Freezers,3.303,7.192,0.000,1.000,67.000,109


## (3) Build and save table

In [10]:
col1_width = "7cm"  # width of variable col
coln_width = "1.5cm"  # width of data cols
indent = r"\hspace{0.3cm} "

def fmt_val(val, dec):
    # Negative sign hangs left via \llap (zero width); positives get a leading space in its
    # place so decimal points stay aligned. "," -> "{,}" since a bare comma in math mode adds spacing.
    if val != val:  # NaN -> blank cell
        return ""
    magnitude = f"{{:,.{dec}f}}".format(abs(val)).replace(",", "{,}")
    sign = r"\llap{-}" if val < 0 else " "
    return f"${sign}{magnitude}$"

col_spec = f"@{{}}L{{{col1_width}}}" + "".join(f"C{{{coln_width}}}" for _ in range(6))
n_cols = 7  # label column + mean, SD, min, median, max, N

lines = []
lines.append(f"\\begin{{tabular}}{{{col_spec}}}")
lines.append(r"\hline")
lines.append(r"\addlinespace[0.2cm]")
lines.append(r" & Mean & SD & Min & Median & Max & N \\")
lines.append(r"\hline")

for panel_label, rows in panels.items():
    lines.append(r"\addlinespace[0.2cm]")
    lines.append(f"\\multicolumn{{{n_cols}}}{{@{{}}l}}{{\\textit{{{panel_label}}}}} \\\\")
    lines.append(r"\addlinespace[0.1cm]")

    for label, var, kind, d_mean, d_range in rows:
        r = stats[stats["label"] == label].iloc[0]
        label_str = f"{indent}\\textbf{{{label}}}" if var == "units_total" else f"{indent}{label}"

        cells = [fmt_val(r["mean"], d_mean)]
        if kind == "cont":
            cells += [fmt_val(r["sd"], d_mean), fmt_val(r["min"], d_range),
                      fmt_val(r["median"], max(d_range, 1)), fmt_val(r["max"], d_range)]
        else:
            cells += [""] * 4
        cells.append(f"${int(r['N']):,}$".replace(",", "{,}"))

        lines.append(f"{label_str} & " + " & ".join(cells) + r" \\")
        lines.append(r"\addlinespace[0.1cm]")

lines.append(r"\addlinespace[0.1cm]")
lines.append(r"\hline")
lines.append(r"\end{tabular}")

table = "\n".join(lines)

In [11]:
out_dir = config.OUTPUT / "14_Summary_Stats"
out_dir.mkdir(parents=True, exist_ok=True)
table_path = out_dir / "summary_stats_table.tex"
_ = table_path.write_text(table)
print(f"Saved: {table_path}")

Saved: /Users/drutna/Dropbox/Apps/Overleaf/UZH Decarb Pre-Analysis Plan/2_Output/14_Summary_Stats/summary_stats_table.tex


In [12]:
print(table)

\begin{tabular}{@{}L{7cm}C{1.5cm}C{1.5cm}C{1.5cm}C{1.5cm}C{1.5cm}C{1.5cm}}
\hline
\addlinespace[0.2cm]
 & Mean & SD & Min & Median & Max & N \\
\hline
\addlinespace[0.2cm]
\multicolumn{7}{@{}l}{\textit{Sample composition}} \\
\addlinespace[0.1cm]
\hspace{0.3cm} Science & $ 0.706$ &  &  &  &  & $109$ \\
\addlinespace[0.1cm]
\hspace{0.3cm} Medicine & $ 0.248$ &  &  &  &  & $109$ \\
\addlinespace[0.1cm]
\hspace{0.3cm} Joint & $ 0.046$ &  &  &  &  & $109$ \\
\addlinespace[0.1cm]
\hspace{0.3cm} Assigned to treatment & $ 0.514$ &  &  &  &  & $109$ \\
\addlinespace[0.1cm]
\hspace{0.3cm} Number of researchers & $ 12.38$ & $ 21.70$ & $ 1$ & $ 8.0$ & $ 214$ & $109$ \\
\addlinespace[0.1cm]
\addlinespace[0.2cm]
\multicolumn{7}{@{}l}{\textit{Baseline annual energy use (MWh)}} \\
\addlinespace[0.1cm]
\hspace{0.3cm} Total & $ 22.03$ & $ 36.95$ & $ 0.23$ & $ 13.31$ & $ 300.97$ & $109$ \\
\addlinespace[0.1cm]
\addlinespace[0.2cm]
\multicolumn{7}{@{}l}{\textit{Baseline equipment (units per group)}} \\
\

## (4) Numbers for the text

Sample counts to quote in the text, for the table's sample (lab groups with both BL and EL data). The treatment/control and faculty counts are also printed for all lab groups surveyed at baseline; institute and enumerator counts are for the table sample only.

In [13]:
# All lab groups surveyed at baseline (the table's sample, bl, is the subset with EL data too)
bl_all = df[df["survey"] == "BL"].drop_duplicates("labgroupid").set_index("labgroupid")

faculty_labels = {
    "Faculty of Science (MNF)": "Science (MNF)",
    "Faculty of Medicine (MeF)": "Medicine (MeF)",
    "Both MNF and MeF": "Joint (MNF and MeF)",
}

def print_sample_numbers(sample, title, institutes_and_enumerators=True):
    print(f"--- {title} ---")
    print(f"Lab groups: {len(sample)}")

    # Treatment / control
    print(f"Treatment: {(sample['treated'] == 1).sum()} | Control: {(sample['treated'] == 0).sum()}")

    # Faculty
    n_by_faculty = {label: (sample["faculty"] == faculty).sum() for faculty, label in faculty_labels.items()}
    assert sum(n_by_faculty.values()) == len(sample) # every lab group is in exactly one faculty group
    print("Faculty: " + " | ".join(f"{label}: {n}" for label, n in n_by_faculty.items()))

    # Institute and enumerator counts are only reported for the table sample
    if not institutes_and_enumerators:
        return

    # Institutes: a singleton institute has exactly one lab group in the sample
    institute_sizes = sample["institute_id"].value_counts()
    print(f"Unique institutes: {len(institute_sizes)} (singleton institutes: {(institute_sizes == 1).sum()})")

    # Enumerators (BL enumerator of each lab group)
    groups_per_enumerator = sample["enum_id"].value_counts()
    print(f"Enumerators: {len(groups_per_enumerator)} | average groups per enumerator: {groups_per_enumerator.mean():.2f}")

print_sample_numbers(bl, "Lab groups with both BL and EL data (table sample)")
print()
print_sample_numbers(bl_all, "All lab groups surveyed at baseline", institutes_and_enumerators=False)

--- Lab groups with both BL and EL data (table sample) ---
Lab groups: 109
Treatment: 56 | Control: 53
Faculty: Science (MNF): 77 | Medicine (MeF): 27 | Joint (MNF and MeF): 5
Unique institutes: 31 (singleton institutes: 11)
Enumerators: 30 | average groups per enumerator: 3.63

--- All lab groups surveyed at baseline ---
Lab groups: 138
Treatment: 70 | Control: 68
Faculty: Science (MNF): 99 | Medicine (MeF): 32 | Joint (MNF and MeF): 7
